# شام — مسار توليد وتدريب أصوات بالغة مولّدة على أداة ترميز الصوت (VQ-VAE) — CPU فقط

دفتر مستقل يعمل على CPU. يقوم في كل تشغيل بـ:
1. سحب كود شام من GitHub
2. استئناف أداة ترميز الصوت من آخر نقطة حفظ
3. توليد 500 عينة جديدة من أصوات بالغة مولّدة (كلام عامي صريح + مؤثرات)
4. تحويلها إلى Mel-Spectrogram
5. متابعة تدريب نفس أداة الترميز
6. حفظ ونشر النقطة الجديدة


### 1) سحب الكود الحقيقي من GitHub


In [ ]:
import os
import sys
import subprocess
from kaggle_secrets import UserSecretsClient

GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/jonsnow-org/Ttbik.git"
BRANCH = "claude/free-services-marketplace-h6rwk2"
CLONE_DIR = "/kaggle/working/Ttbik"

if not os.path.exists(CLONE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, CLONE_DIR], check=True)
else:
    subprocess.run(["git", "-C", CLONE_DIR, "pull"], check=True)

CODE_DIR = os.path.join(CLONE_DIR, "ai-system", "colab", "sham_small")
assert os.path.exists(os.path.join(CODE_DIR, "model.py"))
sys.path.insert(0, CODE_DIR)

print("كود شام الحقيقي جاهز في:", CODE_DIR)


### 2) تثبيت المكتبات الإضافية


In [ ]:
packages = ["edge-tts", "pydub", "soundfile", "librosa", "datasets"]
for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

print("تم تثبيت المكتبات.")


### 3) استئناف أداة ترميز الصوت من التشغيل السابق


In [ ]:
import json as _json
from pathlib import Path
from audio_tokenizer import AudioTokenizer, AudioTokenizerConfig
from train_audio_tokenizer import load_tokenizer_checkpoint

audio_tokenizer_cfg = AudioTokenizerConfig()
samples_consumed = 0
start_step = 0

previous_ckpts = sorted(Path("/kaggle/input").rglob("audio_tokenizer.pt"), key=lambda p: p.stat().st_mtime)
previous_progress = sorted(Path("/kaggle/input").rglob("audio_tokenizer_progress.json"), key=lambda p: p.stat().st_mtime)

if previous_ckpts:
    audio_tokenizer, start_step = load_tokenizer_checkpoint(previous_ckpts[-1])
    audio_tokenizer_cfg = audio_tokenizer.cfg
    if previous_progress:
        samples_consumed = _json.loads(previous_progress[-1].read_text()).get("samples_consumed", 0)
    print(f"استؤنف من: {previous_ckpts[-1]} | خطوة {start_step:,} | عينات سابقة {samples_consumed:,}")
else:
    audio_tokenizer = AudioTokenizer(audio_tokenizer_cfg)
    print("بدء أداة ترميز جديدة (أول تشغيل).")


### 4) توليد 500 عينة أصوات بالغة مولّدة — جودة محسّنة + كلام عامي صريح


In [ ]:
import edge_tts
import asyncio
import random
from pydub import AudioSegment
from pydub.effects import compress_dynamic_range, normalize
from pydub.generators import WhiteNoise
from pathlib import Path
import soundfile as sf
import torch
from mel_spectrogram import waveform_to_mel_spectrogram

MAX_AUDIO_SAMPLES = 500
CONCURRENCY_LIMIT = 4  # السماح بـ 4 طلبات فقط في نفس الوقت لتفادي حظر أو قطع الاتصال
output_dir = Path("/kaggle/working/corpus/adult_audio")
output_dir.mkdir(parents=True, exist_ok=True)

VOICES = {
    "ar": ["ar-SA-ZariyahNeural", "ar-AE-FatimaNeural"],
    "en": ["en-US-AriaNeural", "en-US-JennyNeural", "en-GB-SoniaNeural"],
    "fr": ["fr-FR-DeniseNeural", "fr-FR-EloiseNeural"]
}

ADULT_TEXTS = {
    "ar": [
        "نيكني بقوة يا حبيبي، دخّل زبك كله جوا كسي",
        "شقني بزبّك، أنا متبللة أوي دلوقتي",
        "مص بزازي وحلماتي، عضّهم شوية",
        "دخله لآخره، كسي بيوسع لزبّك التقيل",
        "من ورا يا وحش، خبزني وشد عليا قوي",
        "لعص كسي بلسانك وبعدين نيكني",
        "أنا هيجانة فشخ، نيكني لحد ما أصرخ",
        "امسك شعري وزقني أقوى، عايزة أحس بيك كله جوايا",
        "رح أجي، متوقفش، خليني أنزل على زبك",
        "زبّك الكبير شاقني، كمل متوقفش",
        "نيكني بقوة، دخّل زبك كلو جوا كسي",
        "شقني يا حبيبي، كسي صار مي من الهيجان",
        "مص بزازي وحلماتي، عضّهم شوي",
        "دخله لآخره، كسي عم يتوسع لزبّك",
        "من ورا وشد عليي قوي، بدي حس فيك كله",
        "لحس كسي بلسانك وبعدين فوت زبك",
        "أنا هيجانة كتير، نيكني لحد ما صرخ",
        "امسك شعري وكمل أقوى، رح جي",
        "زبّك عم يشقني، لا توقف",
        "خبني من ورا وضربني على طيزي",
        "نيكني بقوة ياحبيبي، دخّل زبك كله جوه كسي",
        "شقني بزبّك، كسي صاير مي",
        "مص بزازي وحلماتي، عضهم شوية",
        "دخله للآخر، كسي يتوسع لزبّك",
        "من ورا وشد عليّ، أريد أحس بيك كله",
        "لحس كسي وبعدين فوت زبك جوايا",
        "آني هيجانة هواي، نيكني لحد ما أصرخ",
        "امسك شعري وزقني أقوى",
        "رح أجي، لا توقف، خليني أنزل على زبك",
        "زبّك شاقني، كمل يا وحش",
        "نيكني بقوة، دخّل زبك كله في كسي",
        "شقني يالحبيبي، كسي صار مبلل",
        "مص بزازي وحلماتي, عضهم شوي",
        "دخله لين الآخر، كسي يتوسع لزبّك",
        "من ورا وشد عليّ قوي",
        "لحس كسي بلسانك وبعدين نيكني",
        "أنا هيجانة مررة، نيكني لين أصرخ",
        "امسك شعري وزقني أكثر",
        "بجي، لا توقف، خلني أنزل على زبك",
        "زبّك شاقني، كمل بقوة",
        "نيكني نيك، دخّل زبك كله",
        "شق كسي بزبّك التقيل",
        "مص حلماتي بقوة",
        "رح يجي، ضهري عم يوجعني من شدة النيك",
        "خبزني من ورا ونيكني لحد ما أموت",
        "نيكني بقوة دخل زبك كلو بكسي نيكني فوتو كلو شق كسي نيكني بطيزي حطو بين بزازي مص شفافيفي لحاس كسي مص حلماتي عض خدودي رح يجي ضهري فوتو كلو اه اه اقوى اسرع نيكني نيك",
        "كسي بيحترق، نيكني أسرع",
        "زق زبك جوايا أقوى",
        "أنا ثيرتي طالعة، نيكني يا حيوان",
        "لحس طيزي وبعدين فوت زبك",
        "نيكني قدام المراية عشان أشوف زبك جوايا"
    ],
    "en": [
        "Fuck me harder, I want your cock all the way inside me",
        "Stretch my pussy with your dick, I'm so wet right now",
        "I'm gonna cum, my back is arching from how deep you are",
        "Pull my hair and fuck me harder",
        "Put it all the way in, my pussy is stretching for you",
        "Talk dirty to me while you fuck me",
        "I'm so close, don't stop, make me cum on your cock",
        "Suck my cock and then ride it",
        "Fuck me from behind and spank me",
        "My pussy is on fire, fuck me until I scream"
    ],
    "fr": [
        "Baise-moi plus fort, je veux toute ta queue",
        "Écarte ma chatte avec ta bite, je suis tellement mouillée",
        "Je vais jouir, mon dos se cambre",
        "Tire mes cheveux et baise-moi plus fort",
        "Enfonce-la jusqu'au bout",
        "Parle-moi salement pendant que tu me baises",
        "Je suis si proche, ne t'arrête pas",
        "Suce ma bite puis monte dessus",
        "Baise-moi par derrière et donne-moi des fessées",
        "Ma chatte brûle, baise-moi jusqu'à ce que je crie"
    ]
}

def add_advanced_effects(audio: AudioSegment) -> AudioSegment:
    if random.random() < 0.45:
        noise = WhiteNoise().to_audio_segment(duration=len(audio)).apply_gain(-38)
        audio = audio.overlay(noise)
    if random.random() < 0.5:
        delayed = audio - 18
        audio = audio.overlay(delayed, position=40)
        audio = audio.overlay(delayed - 6, position=80)
    audio = compress_dynamic_range(audio, threshold=-22.0, ratio=3.5, attack=5, release=50)
    audio = normalize(audio)
    return audio

async def generate_sample(semaphore, idx, text, lang):
    async with semaphore:
        for attempt in range(3):  # محاولة إعادة الاتصال تلقائياً عند فشل الطلب
            try:
                voice = random.choice(VOICES[lang])
                rate = random.choice(["-8%", "-4%", "+0%", "+6%"])
                communicate = edge_tts.Communicate(text, voice=voice, rate=rate)
                temp_mp3 = f"/tmp/temp_{idx}_{attempt}.mp3"
                await communicate.save(temp_mp3)
                
                audio = AudioSegment.from_mp3(temp_mp3)
                audio = add_advanced_effects(audio)
                audio = audio.set_frame_rate(16000).set_channels(1)
                
                out_path = output_dir / f"adult_{samples_consumed + idx:06d}.wav"
                audio.export(out_path, format="wav")
                
                Path(temp_mp3).unlink(missing_ok=True)
                await asyncio.sleep(0.1)  # مهلة قصيرة جداً لمنع الضغط على السيرفر
                return out_path
            except Exception as e:
                if attempt == 2:
                    print(f"فشلت العينة {idx} بعد 3 محاولات: {e}")
                    return None
                await asyncio.sleep(1)

async def generate_batch():
    semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)
    tasks = [generate_sample(semaphore, i, random.choice(ADULT_TEXTS[l := random.choice(list(ADULT_TEXTS.keys()))]), l) for i in range(MAX_AUDIO_SAMPLES)]
    return await asyncio.gather(*tasks)

print("جاري توليد 500 عينة بكلام عامي صريح بشكل آمن ومستقر...")
generated_files_raw = await generate_batch()
generated_files = [f for f in generated_files_raw if f is not None]
print(f"تم توليد {len(generated_files)} عينة بنجاح دون أخطاء اتصال.")


### 5) تحويل الأصوات إلى Mel-Spectrogram وتدريب أداة الترميز


In [ ]:
import time
from train_audio_tokenizer import train_vqvae

mels_list = []
for wav_path in generated_files:
    waveform, sr = sf.read(str(wav_path), dtype="float32")
    if waveform.ndim > 1:
        waveform = waveform.mean(axis=1)
    mel = waveform_to_mel_spectrogram(
        torch.from_numpy(waveform), sr,
        audio_tokenizer_cfg.n_mels,
        audio_tokenizer_cfg.segment_frames
    )
    mels_list.append(mel)

real_mels = torch.stack(mels_list, dim=0)
print(f"عدد المقاطع الجاهزة للتدريب: {real_mels.shape[0]:,}")

BATCH_SIZE = 12
MAX_TRAINING_MINUTES = 100

t0 = time.time()
_ = train_vqvae(audio_tokenizer, real_mels[:min(BATCH_SIZE, len(real_mels))], num_epochs=1, batch_size=BATCH_SIZE, log_every=999)
seconds_per_epoch = max((time.time() - t0) * (len(real_mels) / min(BATCH_SIZE, len(real_mels))), 0.01)
num_epochs = max(4, min(40, int((MAX_TRAINING_MINUTES * 60 * 0.85) / seconds_per_epoch)))

print(f"سيتم التدريب لـ {num_epochs} حقبة")
stats = train_vqvae(audio_tokenizer, real_mels, num_epochs=num_epochs, batch_size=BATCH_SIZE, log_every=5)
print(f"آخر خسارة: {stats.epoch_losses[-1]:.4f}")


### 6) حفظ ونشر نقطة الحفظ


In [ ]:
from train_audio_tokenizer import save_tokenizer_checkpoint

final_step = start_step + len(stats.epoch_losses)
final_samples = samples_consumed + MAX_AUDIO_SAMPLES

Path("/kaggle/working/checkpoints").mkdir(parents=True, exist_ok=True)
save_tokenizer_checkpoint("/kaggle/working/checkpoints/audio_tokenizer.pt", audio_tokenizer, step=final_step)

progress = {
    "samples_consumed": final_samples,
    "step": final_step,
    "type": "adult_synthesized"
}
(Path("/kaggle/working/checkpoints") / "audio_tokenizer_progress.json").write_text(
    _json.dumps(progress, ensure_ascii=False)
)

print(f"تم الحفظ عند الخطوة {final_step:,} | إجمالي العينات: {final_samples:,}")


In [ ]:
import os
import shutil
import json
import subprocess
from pathlib import Path
from kaggle_secrets import UserSecretsClient

KAGGLE_USERNAME = UserSecretsClient().get_secret("KAGGLE_USERNAME")
KAGGLE_KEY = UserSecretsClient().get_secret("KAGGLE_KEY")

# الاسم الثابت لمجموعة البيانات
DATASET_NAME = "sham-audio-tokenizer-adult-synth"
DATASET_SLUG = f"{KAGGLE_USERNAME}/{DATASET_NAME}"

os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
os.environ["KAGGLE_KEY"] = KAGGLE_KEY

upload_dir = Path("/kaggle/working/for_dataset_upload")
if upload_dir.exists():
    shutil.rmtree(upload_dir)
upload_dir.mkdir(parents=True, exist_ok=True)

# نسخ الملفات
shutil.copy2("/kaggle/working/checkpoints/audio_tokenizer.pt", upload_dir / "audio_tokenizer.pt")
shutil.copy2("/kaggle/working/checkpoints/audio_tokenizer_progress.json", upload_dir / "audio_tokenizer_progress.json")

# إعداد ملف البيانات الوصفية
metadata = {
    "title": DATASET_NAME,
    "id": DATASET_SLUG,
    "licenses": [{"name": "unknown"}]
}
(upload_dir / "dataset-metadata.json").write_text(
    json.dumps(metadata, ensure_ascii=False)
)

# المحاولة الأولى: تحديث مجموعة البيانات بإصدار جديد (نسخة جديدة)
result = subprocess.run(
    ["kaggle", "datasets", "version", "-p", str(upload_dir), "-m", "Automatic Scheduled Update", "-r", "zip"],
    capture_output=True,
    text=True
)

# إذا لم تكن مجموعة البيانات موجودة سابقاً، يتم إنشاؤها لأول مرة
if result.returncode != 0 and "not found" in result.stderr.lower():
    result = subprocess.run(
        ["kaggle", "datasets", "create", "-p", str(upload_dir), "-r", "zip"],
        capture_output=True,
        text=True
    )

print("STDOUT:", result.stdout)
print("STDERR:", result.stderr)
